# Transcribe audio 

Transcribes all the videos in given folder, then attributes and stitches together transcripts based on the saved recording ids.
If recording IDs get missed, we don't stitch them into the final version, so maybe should write a check for that.

In [1]:
! pip install --upgrade deepgram-sdk
! pip install boto3
! pip install python-dotenv

  Using cached deepgram_sdk-4.5.0-py3-none-any.whl.metadata (26 kB)
Using cached deepgram_sdk-4.5.0-py3-none-any.whl (156 kB)
  Attempting uninstall: deepgram-sdk
    Found existing installation: deepgram-sdk 4.1.0
    Uninstalling deepgram-sdk-4.1.0:
      Successfully uninstalled deepgram-sdk-4.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 18.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.34.69
    Uninstalling botocore-1.34.69:
      Successfully uninstalled botocore-1.34.69
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.39.4 which is incompatible.


In [9]:
import os
import glob
import re
import pandas as pd
import json
from dotenv import load_dotenv
from deepgram import ( DeepgramClient , FileSource, PrerecordedOptions )
import shutil

load_dotenv()

recordings_folder = 'recordings'

API_KEY = os.getenv("DG_API_KEY")
deepgram = DeepgramClient(API_KEY)

In [4]:
# transcribe files and create utterances and words csvs for each audio file
audio_files = glob.glob(f'recordings/2025*_pilot*/*audio*.webm', recursive=True)

for audio_file in audio_files:
    utterances_filepath = audio_file.replace(".webm", "-utterances.csv")
    words_filepath = audio_file.replace(".webm", "-words.csv")
    if not os.path.exists(utterances_filepath):
        print("===================================================================================")
        print("transcribing", audio_file)
        
        # get time offset by filepath timestamp
        group_start = int(audio_file.split("/")[-1].split('-')[0])
        filepath_start = int(audio_file.split('-')[-1].split('.')[0])
        filepath_offset = (filepath_start - group_start)/1000

        # get time offset by audio file internal start time
        output = ! ffprobe $audio_file
        start_time_match = re.search(' start: (\d+\.\d+)', '\n'.join(output))
        timestamp_offset = float(start_time_match.group(1)) if start_time_match else None

        if abs(timestamp_offset - filepath_offset) > 1:
            print(f"Warning: timestamp_offset {timestamp_offset} and filepath_offset {filepath_offset} are more than 1 second apart")
    
        with open(audio_file, "rb") as file:
            buffer_data = file.read()

        payload: FileSource = {
            "buffer": buffer_data,
        }

        #STEP 2: Configure Deepgram options for audio analysis
        options = PrerecordedOptions(
            model="nova-2",
            utterances=True,
        )

        # STEP 3: Call the transcribe_file method with the text payload and options
        response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)


        if len(response.to_dict()['results']['utterances']) == 0:
            print("no words in", audio_file)
            pd.DataFrame().to_csv(utterances_filepath, index=False) # create empty file
            continue


        transcription_df = pd.DataFrame(response.to_dict()['results']['utterances'])

        # export utterances csv
        utterances = transcription_df[["start", "end", "confidence", "transcript"]]
        utterances = utterances.rename(columns={"start": "raw_start", "end": "raw_end"})

        utterances["filepath_start"] = utterances["raw_start"] + filepath_offset
        utterances["filepath_end"] = utterances["raw_end"] + filepath_offset

        utterances['timestamp_start'] = utterances['raw_start'] + timestamp_offset
        utterances['timestamp_end'] = utterances['raw_end'] + timestamp_offset
        utterances.to_csv(utterances_filepath, index=False)

        # export words csv
        words_list = []
        for j, word_row in transcription_df.iterrows():
            words_list += word_row['words']

        words = pd.DataFrame(words_list)
        words = words.rename(columns={"start": "raw_start", "end": "raw_end"})

        words['filepath_start'] = words['raw_start'] + filepath_offset
        words['filepath_end'] = words['raw_end'] + filepath_offset

        words['timestamp_start'] = words['raw_start'] + timestamp_offset
        words['timestamp_end'] = words['raw_end'] + timestamp_offset
        words.to_csv(words_filepath, index=False)

transcribing recordings/20250708_1813_pilotTD1HQA/1752000906031-875ed072-9a27-42db-9239-856bbd5311b9-cam-audio-1752000907087.webm


<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_98666/3309302444.py:18: SyntaxWarning: invalid escape sequence '\d'
  start_time_match = re.search(' start: (\d+\.\d+)', '\n'.join(output))
/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_98666/3309302444.py:38: UnsupportedWarning: prerecorded is unsupported as of 4.0.0. deepgram.listen.prerecorded is deprecated. Use deepgram.listen.rest instead.
  response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)


transcribing recordings/20250708_1813_pilotTD1HQA/1751999778086-7dc250b2-b0dd-4d0d-a829-f67b47cc9233-cam-audio-1751999779141.webm


/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_98666/3309302444.py:38: UnsupportedWarning: prerecorded is unsupported as of 4.0.0. deepgram.listen.prerecorded is deprecated. Use deepgram.listen.rest instead.
  response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)


transcribing recordings/20250708_1813_pilotTD1HQA/1752000325421-1b6675b2-0ac4-48c6-bac5-1fbc37a98fe1-cam-audio-1752000326792.webm


...

/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_98666/3309302444.py:38: UnsupportedWarning: prerecorded is unsupported as of 4.0.0. deepgram.listen.prerecorded is deprecated. Use deepgram.listen.rest instead.
  response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)


transcribing recordings/20250317_1538_pilotA3Q9H1Q/1742230731484-0a89d2e5-4b9d-4a76-8ef2-8f0f7354f44e-cam-audio-1742230732322.webm


/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_98666/3309302444.py:38: UnsupportedWarning: prerecorded is unsupported as of 4.0.0. deepgram.listen.prerecorded is deprecated. Use deepgram.listen.rest instead.
  response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)


no words in recordings/20250317_1538_pilotA3Q9H1Q/1742230731484-0a89d2e5-4b9d-4a76-8ef2-8f0f7354f44e-cam-audio-1742230732322.webm
transcribing recordings/20250317_1538_pilotA3Q9H1Q/1742230285598-53647a7b-2e40-46db-a46e-f67bb7c07125-cam-audio-1742230286399.webm


/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_98666/3309302444.py:38: UnsupportedWarning: prerecorded is unsupported as of 4.0.0. deepgram.listen.prerecorded is deprecated. Use deepgram.listen.rest instead.
  response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)


In [6]:
# Load the experiment data so we can assign positions and game ids to each utterance

# pd.options.display.max_rows = None
recordings = pd.read_csv('recordings/recordings_list.csv')
recordings.head().T



,0,1,2,3,4
path,recordings/20250618_1821_pilot34BR1P/175027195...,recordings/20250618_1821_pilot34BR1P/175027195...,recordings/20250618_1821_pilot34BR1P/175027195...,recordings/20250618_1821_pilot34BR1P/175027195...,recordings/20250618_1821_pilot34BR1P/175027195...
folder,20250618_1821_pilot34BR1P,20250618_1821_pilot34BR1P,20250618_1821_pilot34BR1P,20250618_1821_pilot34BR1P,20250618_1821_pilot34BR1P
filename,1750271953804-47150ec5-6c1c-4cc1-abca-2772bf3a...,1750271953804-47150ec5-6c1c-4cc1-abca-2772bf3a...,1750271953804-6e0b8055-3e36-4c10-b029-5baf6d0b...,1750271953804-6e0b8055-3e36-4c10-b029-5baf6d0b...,1750271953804-d83b1ff4-a831-4418-9a7c-1cc640eb...
date,20250618,20250618,20250618,20250618,20250618
time,1821,1821,1821,1821,1821
mtgStartTime,1750271953804,1750271953804,1750271953804,1750271953804,1750271953804
sessionId,47150ec5_6c1c_4cc1_abca_2772bf3a223d,47150ec5_6c1c_4cc1_abca_2772bf3a223d,6e0b8055_3e36_4c10_b029_5baf6d0be6fb,6e0b8055_3e36_4c10_b029_5baf6d0be6fb,d83b1ff4_a831_4418_9a7c_1cc640ebe4ea
group,pilot34BR1P,pilot34BR1P,pilot34BR1P,pilot34BR1P,pilot34BR1P
filetype,audio,video,audio,video,audio
gameId,01JY25CKAD4BKEXHF4Q934BR1P,01JY25CKAD4BKEXHF4Q934BR1P,01JY25CKAD4BKEXHF4Q934BR1P,01JY25CKAD4BKEXHF4Q934BR1P,01JY25CKAD4BKEXHF4Q934BR1P


# Compile utterances csv for each group

In [22]:
# Define the destination folder for finalized files
READY_TO_CODE_DIR = "recordings/ready_to_code"

# Build overall utterances csv for each game
for game_index, group in recordings.groupby(["group", "mtgStartTime"]):
    conversation_utterances = pd.DataFrame()
    group_folder = f"{recordings_folder}/{group['folder'].iloc[0]}"
    for i, row in group.iterrows():
        if row['filetype'] != 'audio':
            continue

        try:
            utterances_df = pd.read_csv(f"{group_folder}/{row['filename']}".replace(".webm", "-utterances.csv"))
            utterances_df["conversation_id"] = row['mtgStartTime']
            utterances_df["position"] = row['position']

            conversation_utterances = pd.concat([conversation_utterances, utterances_df])
        except Exception as e:
            print(f"Error processing {row['filename']}: {e}")
            continue

    if conversation_utterances.shape[0] == 0:
        print(f"❌ no utterances found for folder {group_folder}")
        continue

    try:
        conversation_utterances.sort_values(by=['timestamp_start'], inplace=True)
        output_path = f"{group_folder}/{row['mtgStartTime']}_group-utterances.csv"
        conversation_utterances.to_csv(output_path, index=False)

        # Copy the finalized file
        date, time, group_id = group_folder.split("/")[-1].split("_")
        new_filename = f"{date}_{time}_group_{group_id}_{row['mtgStartTime']}_utterances.csv"
        destination_path = os.path.join(READY_TO_CODE_DIR, new_filename)
        shutil.copy(output_path, destination_path)
        print(f"👉 Copied to: {destination_path}")
    except:
        print("problem with group", group_id)




👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2BY1T9_1752000153821_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2BY1T9_1752000618158_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2BY1T9_1752000980607_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2BY1T9_1752001203273_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2XT15E_1752000324437_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2XT15E_1752000597133_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2XT15E_1752000786956_utterances.csv
👉 Copied to: recordings/ready_to_code/20250708_1813_group_pilot2XT15E_1752000924867_utterances.csv
Error processing 1750271953804-6e0b8055-3e36-4c10-b029-5baf6d0be6fb-cam-audio-1750271961404.webm: No columns to parse from file
Error processing 1750271953804-d83b1ff4-a831-4418-9a7c-1cc640ebe4ea-cam-audio-17

In [16]:
conversation_utterances

,raw_start,raw_end,confidence,transcript,filepath_start,filepath_end,timestamp_start,timestamp_end,conversation_id,position
0,0.880000,1.38000,0.948284,hello,1.968000,2.46800,1.928000,2.42800,1752000153821,0
0,2.000000,2.50000,0.655738,oh,3.094000,3.59400,3.045000,3.54500,1752000153821,1
1,3.040000,3.54000,0.994660,okay,4.134000,4.63400,4.085000,4.58500,1752000153821,1
2,5.120000,7.94000,0.979093,so we have to come up with labels for each of ...,6.214000,9.03400,6.165000,8.98500,1752000153821,1
3,11.759999,15.70000,0.981830,so the name should be short easy to remember a...,12.853999,16.79400,12.804999,16.74500,1752000153821,1
...,...,...,...,...,...,...,...,...,...,...
48,256.610020,258.15000,0.945134,and bigger okay,257.704020,259.24400,257.655020,259.19500,1752000153821,1
49,259.010000,261.03000,0.989105,what do you think go to the,260.104000,262.12400,260.055000,262.07500,1752000153821,1
50,261.570000,262.47000,0.974321,go to the quiz,262.664000,263.56400,262.615000,263.51500,1752000153821,1
38,262.624940,264.88495,0.900448,yeah let's go through the quiz good luck,263.712940,265.97295,263.672940,265.93295,1752000153821,0


In [ ]:
# Define the destination folder for finalized files
READY_TO_CODE_DIR = "recordings/ready_to_code"

# Build overall utterances csv for each game
for game_index, group in data.groupby(["group", "mtgStartTime"]):
    conversation_utterances = pd.DataFrame()
    for i, participant in group.iterrows():
        folder = f"{recordings_folder}/{participant['recordingsFolder']}"
        recordingIds = participant["recordingIds"]
        if "missing" in recordingIds: 
            print(f"❌ missing recordings for {participant['recordingsFolder']}")
            continue

        recordingIds = json.loads(participant["recordingIds"].replace("'", '"'))

        files = []
        for i, recordingId in enumerate(recordingIds):
            filesFound = glob.glob(f"{folder}/*{recordingId}*-utterances.csv")
            if len(filesFound) == 0:
                print(f"❌ no files match {recordingId} in folder {participant['recordingsFolder']}")
            elif len(filesFound) > 1:
                # print(f"multiple files found for {recordingId} for {participant['recordingsFolder']}")
                files += filesFound
            else:
                files.append(filesFound[0])

        participant_utterances = pd.DataFrame()
        for file in files:
            try:
                utterances_df = pd.read_csv(file)
                participant_utterances = pd.concat([participant_utterances, utterances_df])
                participant_utterances['file'] = file
            except Exception as e:
                if "No columns to parse from file" in str(e):
                    continue
                print("❌ error in", file, e)
        participant_utterances['conversation_id'] = participant['gameId']
        participant_utterances['position'] = participant['position']
        participant_utterances['speaker'] = participant['deliberationId']
        
        conversation_utterances = pd.concat([conversation_utterances, participant_utterances])
    
    if conversation_utterances.shape[0] == 0:
        print(f"❌ no utterances found for folder {folder}")
        continue

    try:
        conversation_utterances.sort_values(by=['timestamp_start'], inplace=True)
        output_path = f"{folder}/group-utterances.csv"
        conversation_utterances.to_csv(output_path, index=False)

        # Copy the finalized file
        date, time, batch, group_id = folder.split("/")[-1].split("_")
        new_filename = f"{date}_{time}_batch_{batch}_group_{group_id}_game_{group['game'].iloc[0]}_utterances.csv"
        destination_path = os.path.join(READY_TO_CODE_DIR, new_filename)
        shutil.copy(output_path, destination_path)
        print(f"👉 Copied to: {destination_path}")
    except:
        print("problem with recordings", recordingIds)


👉 Copied to: recordings/ready_to_code/20241001_1844_batch_ctt_group_w1V35R4E_game_0_utterances.csv
👉 Copied to: recordings/ready_to_code/20241004_1934_batch_ctt_group_w1M7AWTB_game_1_utterances.csv
👉 Copied to: recordings/ready_to_code/20240923_1531_batch_CTT_group_W1GY0Y4B_game_2_utterances.csv
👉 Copied to: recordings/ready_to_code/20241009_1402_batch_ctt_group_w1K8S0X5_game_3_utterances.csv
👉 Copied to: recordings/ready_to_code/20241001_1844_batch_ctt_group_w1143Q1A_game_4_utterances.csv
👉 Copied to: recordings/ready_to_code/20240920_1732_batch_CT_group_topDE1YJE_game_5_utterances.csv
👉 Copied to: recordings/ready_to_code/20241023_1936_batch_ctt_group_w151PGAV_game_6_utterances.csv
👉 Copied to: recordings/ready_to_code/20241023_1611_batch_ctt_group_w1513W2X_game_7_utterances.csv
👉 Copied to: recordings/ready_to_code/20241001_1844_batch_ctt_group_w1W06WAX_game_8_utterances.csv
👉 Copied to: recordings/ready_to_code/20241028_1752_batch_ctt_group_w16DGSYJ_game_9_utterances.csv
👉 Copied t

# Compile words csv for each game

In [36]:
READY_TO_CODE_DIR = "recordings/ready_to_code"

# build overall words csv for each game
for game_index, group in data.groupby("game"):
    # try:
        conversation_words = pd.DataFrame()
        for i, participant in group.iterrows():
            folder = f"{recordings_folder}/{participant['recordingsFolder']}"
            raw_recordingIds = participant["recordingIds"]
            if "missing" in raw_recordingIds:
                print("❌ missing in", folder)
                continue
            recordingIds = json.loads(participant["recordingIds"].replace("'", '"'))
            
            files = []
            for recordingId in recordingIds:
                glob_files = glob.glob(f"{folder}/*{recordingId}*-words.csv")
                if len(glob_files) == 0:
                    print(f"❌ no files match {recordingId} in folder {participant['recordingsFolder']}")
                elif len(glob_files) > 1:
                    print(f"❌ multiple files found for {recordingId} for {participant['recordingsFolder']}")
                else:
                    files.append(glob_files[0])

            # files = [glob.glob(f"{folder}/*{recordingId}*-words.csv")[0] for recordingId in recordingIds]
            participant_words = pd.DataFrame()
            for file in files:
                try:
                    words_df = pd.read_csv(file)
                    participant_words = pd.concat([participant_words, words_df])
                    participant_words['file'] = file
                except:
                    print("❌ error in", file)
            participant_words['conversation_id'] = participant['gameId']
            participant_words['position'] = participant['position']
            
            conversation_words = pd.concat([conversation_words, participant_words])

        if conversation_words.shape[0] == 0:
            print("❌ no words in", folder)
            continue
        conversation_words.sort_values(by=['timestamp_start'], inplace=True)

        output_path = f"{folder}/group-words.csv"
        conversation_words.to_csv(output_path, index=False)
        print("✅", folder)

        # Copy the finalized file
        date, time, batch, group_id = folder.split("/")[-1].split("_")
        new_filename = f"{date}_{time}_batch_{batch}_group_{group_id}_game_{group['game'].iloc[0]}_words.csv"
        destination_path = os.path.join(READY_TO_CODE_DIR, new_filename)
        shutil.copy(output_path, destination_path)
        print(f"👉 Copied to: {destination_path}")
    # except Exception as e:
        # print("error in", folder, e)
        # print("❌", folder)
        # break

✅ recordings/20241001_1844_ctt_w1V35R4E
👉 Copied to: recordings/ready_to_code/20241001_1844_batch_ctt_group_w1V35R4E_game_0_words.csv
✅ recordings/20241004_1934_ctt_w1M7AWTB
👉 Copied to: recordings/ready_to_code/20241004_1934_batch_ctt_group_w1M7AWTB_game_1_words.csv
❌ no files match a1efd28a-af9d-4217-b785-204f7547f268 in folder 20240923_1531_CTT_W1GY0Y4B
✅ recordings/20240923_1531_CTT_W1GY0Y4B
👉 Copied to: recordings/ready_to_code/20240923_1531_batch_CTT_group_W1GY0Y4B_game_2_words.csv
✅ recordings/20241009_1402_ctt_w1K8S0X5
👉 Copied to: recordings/ready_to_code/20241009_1402_batch_ctt_group_w1K8S0X5_game_3_words.csv
✅ recordings/20241001_1844_ctt_w1143Q1A
👉 Copied to: recordings/ready_to_code/20241001_1844_batch_ctt_group_w1143Q1A_game_4_words.csv
✅ recordings/20240920_1732_CT_topDE1YJE
👉 Copied to: recordings/ready_to_code/20240920_1732_batch_CT_group_topDE1YJE_game_5_words.csv
✅ recordings/20241023_1936_ctt_w151PGAV
👉 Copied to: recordings/ready_to_code/20241023_1936_batch_ctt_gro

In [34]:
conversation_words

,conversation_id,position


# Try to understand missing recordings

In [12]:
recording_stats = []
for game_index, group in data.groupby("game"):
    for i, participant in group.iterrows():
        folder = f"{recordings_folder}/{participant['recordingsFolder']}"
        stats = {
            "game": game_index,
            "participant": i,
            "folder": folder,
            "folder_exists": os.path.exists(folder),
        }

        recording_stats.append(stats)

recording_stats_df = pd.DataFrame(recording_stats)
recording_stats_df

,game,participant,folder,folder_exists
0,0,0,recordings/20241001_1844_ctt_w1V35R4E,True
1,0,1,recordings/20241001_1844_ctt_w1V35R4E,True
2,1,2,recordings/20241004_1934_ctt_w1M7AWTB,True
3,1,3,recordings/20241004_1934_ctt_w1M7AWTB,True
4,2,4,recordings/20240923_1531_CTT_W1GY0Y4B,True
...,...,...,...,...
1026,513,1026,recordings/20241217_1837_w2s19_BS9KEY,True
1027,514,1027,recordings/20241219_1734_w2s26_19WP2C,True
1028,514,1028,recordings/20241219_1734_w2s26_19WP2C,True
1029,515,1029,recordings/20241217_1837_w2s19_RFCWYF,True


In [13]:
data.head()

,treatment,game,position,batchId,batchName,deliberationId,sampleId,gameId,recordingIds,recordingsFolder,...,partner_race_Native Hawaiian or other Pacific Islander,party_difference,years_seniority,guessPartnerPartyError,guessPartnerPartyExtremity,guessPartnerTopicError,guessPartnerTopicExtremity,overall_demographic_difference,duration_Part1,duration_Part2
0,wave_1_fundNeedleExchange_dem_0_rep_0,0,0,01J94PCAWCN6AS2Q7K9GHF45KV,ctt_w1_s6_CR,4c923fdd-fd15-4780-9f16-5a60e040def6,667c2bfe-8401-4272-9ea6-44f0aef7c38c,01J94SKS2GJ71CT03G7XV35R4E,['252fcba0-f6df-41af-ae27-3516965e50b4'],20241001_1844_ctt_w1V35R4E,...,0.0,1.000,-4.0,0.00000,0.50000,0.000000,0.500000,5.991534,1466,1680.0
1,wave_1_fundNeedleExchange_dem_0_rep_0,0,1,01J94PCAWCN6AS2Q7K9GHF45KV,ctt_w1_s6_CR,a125d806-2bcf-496a-a9ae-f727a2803d2f,0be567b7-cda5-42eb-9098-c0187b0225c3,01J94SKS2GJ71CT03G7XV35R4E,['c3eac583-9b28-4756-8cc6-947d2fc09e31'],20241001_1844_ctt_w1V35R4E,...,0.0,1.000,4.0,0.16666,0.33334,0.500000,0.000000,5.991534,382,929.0
2,wave_1_fundNeedleExchange_dem_0_rep_5,1,0,01J9CGDS649P6NV967E7YCNXS8,ctt_w1_s11_CR,4d8ce909-c9ec-411d-9fa5-9ea02bf0e0b5,3fe79b64-0703-4a0e-8389-878ba6f5a386,01J9CJRDKZCGRP8NWM0RM7AWTB,['1cfdf05f-0eb7-4311-bce8-fbaca8f229a5'],20241004_1934_ctt_w1M7AWTB,...,0.0,0.833,-21.0,0.16634,0.16666,0.333333,0.000000,6.467071,256,882.0
3,wave_1_fundNeedleExchange_dem_0_rep_5,1,1,01J9CGDS649P6NV967E7YCNXS8,ctt_w1_s11_CR,d3b80e89-be64-49a0-a26b-1c43b8aec6df,b9c46cfa-6b1d-4f62-8efa-2362ac93dfe1,01J9CJRDKZCGRP8NWM0RM7AWTB,['eb2785d2-ee20-488a-9bd8-d8da06724910'],20241004_1934_ctt_w1M7AWTB,...,0.0,0.833,21.0,0.00000,0.50000,0.000000,0.500000,6.467071,319,954.0
4,wave_1_fundNeedleExchange_dem_0_rep_6,2,0,01J8FR4S38XRPMEGQSJ4WGVSTX,CTT_W1_S3,3d46635f-680a-470e-a835-29f826a9471e,2bd517af-2e4c-41f8-9dea-ee2eff4b2ca4,01J8FVGE4701CXJQ35AHGY0Y4B,"['299867da-d96e-4ad4-9f41-e3033ac95261', '79d5...",20240923_1531_CTT_W1GY0Y4B,...,0.0,0.833,-18.0,0.00033,0.33333,0.333333,0.166667,5.007841,638,1763.0


In [14]:
recording_stats_df["folder_exists"].value_counts()

folder_exists
True     1025
False       6
Name: count, dtype: int64

In [15]:
data['recordings_downloaded'] = data['recordingsFolder'].apply(lambda x: os.path.exists(f"{recordings_folder}/{x}"))
data['recordings_downloaded'].value_counts()

recordings_downloaded
True     1025
False       6
Name: count, dtype: int64

In [16]:
data[(data['discussion_happened'] == True) & (data['recordings_downloaded'] == False)]

,treatment,game,position,batchId,batchName,deliberationId,sampleId,gameId,recordingIds,recordingsFolder,...,party_difference,years_seniority,guessPartnerPartyError,guessPartnerPartyExtremity,guessPartnerTopicError,guessPartnerTopicExtremity,overall_demographic_difference,duration_Part1,duration_Part2,recordings_downloaded
569,wave_2_capAndTrade_dem_6_rep_0,285,0,01JEY2M6EVFAZ4WGZWP29BHSV8,w2s12_ctt_prolific,2370bb1e-7e95-499c-8fa4-c0bb4bd86030,a24cd8c8-14ca-440c-a7c2-0f02a2210267,01JEY3WNNTDPYH8RA1V3J5C3YN,['9d59e259-b76f-49f1-8adc-98d53cc1f1c7'],20241212_1838_w2s12_J5C3YN,...,1.0,8.0,NaN,NaN,NaN,NaN,4.892180,346,NaN,False
570,wave_2_capAndTrade_dem_6_rep_0,285,1,01JEY2M6EVFAZ4WGZWP29BHSV8,w2s12_ctt_prolific,14d93321-79cd-4798-a5cd-92f4d296115d,bf7c4436-2fc8-408a-b387-11d63ef69455,01JEY3WNNTDPYH8RA1V3J5C3YN,['be04edee-ff60-43fd-a8bd-44851d0e0af6'],20241212_1838_w2s12_J5C3YN,...,1.0,-8.0,0.00000,0.50000,0.333333,0.166667,4.892180,320,856.0,False
671,wave_2_cubaEmbargo_dem_3_rep_5,336,0,01JEY9A5H2086RQ622BE75Z2G2,w2s14_ctt_prolific,6627c9cc-2e2e-4080-af5e-287acefea3e6,f4198256-96f3-4699-ae71-2f58a815e872,01JEYARC27QPQ6WQEFWYDB7Q51,['0b44e767-f403-497a-ab70-2bac49e3af78'],20241212_2035_w2s14_DB7Q51,...,0.5,-12.0,0.16634,0.16666,0.166667,0.166667,5.184515,321,845.0,False
672,wave_2_cubaEmbargo_dem_3_rep_5,336,1,01JEY9A5H2086RQ622BE75Z2G2,w2s14_ctt_prolific,8dbdfad9-1ef8-4456-b553-d5a9a9f58167,ced582ea-2938-4640-83b5-dcbc791e76b1,01JEYARC27QPQ6WQEFWYDB7Q51,['3f2202a7-4b7f-49db-9b51-87b82297aef9'],20241212_2035_w2s14_DB7Q51,...,0.5,12.0,0.00033,0.16667,0.333333,0.333333,5.184515,392,925.0,False


In [17]:
data.columns

Index(['treatment', 'game', 'position', 'batchId', 'batchName',
       'deliberationId', 'sampleId', 'gameId', 'recordingIds',
       'recordingsFolder',
       ...
       'party_difference', 'years_seniority', 'guessPartnerPartyError',
       'guessPartnerPartyExtremity', 'guessPartnerTopicError',
       'guessPartnerTopicExtremity', 'overall_demographic_difference',
       'duration_Part1', 'duration_Part2', 'recordings_downloaded'],
      dtype='object', length=279)